# 06-1. 네트워크와 소켓 기초 예제

## Goal

- IP 주소와 포트를 하나의 엔드포인트로 검증합니다.
- 주소 체계와 소켓 종류를 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

Python 표준 라이브러리만 사용하며 외부 네트워크에 연결하지 않습니다.


## Steps

### 루프백 엔드포인트 검증

포트의 자료형과 범위를 먼저 확인하고, 학습 대상 주소를 루프백으로 제한합니다.


In [1]:
from dataclasses import dataclass
import ipaddress
import socket


@dataclass(frozen=True)
class Endpoint:
    host: str
    port: int
    family: socket.AddressFamily


def validate_endpoint(host: str, port: int) -> Endpoint:
    if not isinstance(host, str) or not host:
        raise TypeError("host는 비어 있지 않은 문자열이어야 합니다")
    if isinstance(port, bool) or not isinstance(port, int):
        raise TypeError("port는 정수여야 합니다")
    if not 1 <= port <= 65535:
        raise ValueError("port는 1~65535 범위여야 합니다")
    address = ipaddress.ip_address(host)
    if not address.is_loopback:
        raise ValueError("이 실습은 루프백 주소만 허용합니다")
    family = socket.AF_INET6 if address.version == 6 else socket.AF_INET
    return Endpoint(str(address), port, family)


endpoint = validate_endpoint("127.0.0.1", 9000)
print(endpoint)
print("주소 체계:", endpoint.family.name)
print("스트림 소켓:", socket.SocketKind.SOCK_STREAM.name)


Endpoint(host='127.0.0.1', port=9000, family=<AddressFamily.AF_INET: 2>)
주소 체계: AF_INET
스트림 소켓: SOCK_STREAM


## Checks

정상값과 경계값을 검증합니다. `bool`은 `int`의 하위형이므로 별도로 거부합니다.


In [2]:
assert validate_endpoint("::1", 443).family == socket.AF_INET6
for bad_port in (True, 0, 65536):
    try:
        validate_endpoint("127.0.0.1", bad_port)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"거부되지 않은 포트: {bad_port!r}")
print("경계값 검사 통과")


경계값 검사 통과


## Next Steps

실제 TCP 연결은 06-2와 06-3의 클라이언트·서버 파일에서 수행합니다.
